# Voltage decay across a neuron using passive conductances
## Modeled in ngspice via a cable theory framework
#### http://ngspice.sourceforge.net/ (see the simulation.py file in neuprint-python for directions on setting up ngspice)
#### Modified from Scheffer et al. A Connectome and Analysis of the Adult Drosophila Central Brain
#### https://www.biorxiv.org/content/10.1101/2020.04.07.030213v1
#### With personal thanks to Lou Scheffer, Stuart Berg, and Steve Plaze for their help
#### Also see:
- Segev et. al. Modeling the Electrical Behavior of Anatomically Complex Neurons Using a Network Analysis Program: Passive Membrane
    - https://link.springer.com/article/10.1007/BF00355688
- Gornet and Scheffer. Simulating extracted connectomes
    - https://www.biorxiv.org/content/10.1101/177113v2
- Gouwens and Wilson. Signal Propagation in Drosophila Central Neurons
    - https://www.jneurosci.org/content/29/19/6239

## Load libraries and setup NeuPrint client

In [ ]:
from IPython.display import Image
from neuprint import Client
from neuprint import fetch_neurons, fetch_synapses, fetch_synapse_connections, fetch_adjacencies
from neuprint import merge_neuron_properties
from neuprint import NeuronCriteria as NC, SynapseCriteria as SC
from neuprint import NeuronModel
from neuprint.utils import tqdm, UMAP
from neuprint.client import inject_client
import neuprint as neu
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
import navis.interfaces.neuprint as neu
#import mplcursors

import matplotlib.pyplot as plt
import matplotlib.patches as mplpatches
import bokeh
import hvplot.pandas
import holoviews as hv
import bokeh.palettes
from bokeh.plotting import figure, show, output_notebook, ColumnDataSource
from bokeh.themes import built_in_themes
from bokeh.io import curdoc, output_file, export_png
from bokeh.layouts import gridplot
from bokeh.models import LinearColorMapper, ColorBar

#import mplcursors 

import navis

import time
import os
from tempfile import mkstemp
from subprocess import Popen, PIPE, DEVNULL

# --- NeuPrint credentials --------------------------------------------------
# Never hard-code a token. Get a free one at https://neuprint.janelia.org
# (Account -> Auth Token) and expose it to this notebook as an env variable:
#     PowerShell:  $env:NEUPRINT_TOKEN = "<your-token>"
#     bash:        export NEUPRINT_TOKEN="<your-token>"
NEUPRINT_TOKEN = (os.environ.get("NEUPRINT_TOKEN")
                  or os.environ.get("NEUPRINT_APPLICATION_CREDENTIALS"))
if not NEUPRINT_TOKEN:
    raise RuntimeError(
        "No NeuPrint token found. Set NEUPRINT_TOKEN (or "
        "NEUPRINT_APPLICATION_CREDENTIALS) in your environment before running "
        "this notebook -- see the Credentials section of the README."
    )

c = Client('neuprint.janelia.org',
           dataset='hemibrain:v1.2.1',
           token=NEUPRINT_TOKEN)
c.fetch_version()


In [ ]:
#pip install bokeh==3.4.0

#### Add ngspice bin folder location

In [ ]:
# Location of the ngspice `bin` directory (the one holding ngspice_con.exe
# on Windows, or ngspice on macOS/Linux). Set it per machine:
#     PowerShell:  $env:NGSPICE_BIN = "C:\\ngspice\\Spice64\\bin"
#     bash:        export NGSPICE_BIN="/usr/local/bin"
# Only needed when RUN_SIMULATION is True.
SPICE_FOLDER = os.environ.get("NGSPICE_BIN", "")

if SPICE_FOLDER and os.path.isdir(SPICE_FOLDER):
    print(f"ngspice bin: {SPICE_FOLDER}")
else:
    print("NGSPICE_BIN is unset or not a directory -- the sweep cannot run.\n"
          "Leave RUN_SIMULATION = False to work from committed results instead.")


In [ ]:
# --- Repository paths, readout region, and run mode -------------------------
from pathlib import Path

# Portable repo-root bootstrap: walk up until we find the repo layout, so the
# notebook runs from anywhere in the tree (no hard-coded drive letters).
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'results').is_dir() and (p / 'notebooks').is_dir())
RESULTS_DIR = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
print(f"repo root: {REPO_ROOT}")

# Which readout region THIS notebook monitors. The sister notebook uses the
# other value, and every output file is tagged with it, so the two notebooks
# can never overwrite each other's results.
#   'IDRandAxonVDR_region' -> IDR + axon NaV tract (~73 nodes)
#   'AxonVDR_region'       -> axon / DAS only      (~50 nodes)
REGION = 'AxonVDR_region'

# True  -> run the full ngspice sweep (one transient analysis per input synapse;
#          ~10,494 of them -- hours of compute, and needs NGSPICE_BIN set).
# False -> skip the sweep and read results from disk (newest matching run in
#          results/, else the committed reduced CSV for this region).
RUN_SIMULATION = False

COMMITTED_RESULTS = {
    'IDRandAxonVDR_region': RESULTS_DIR / 'idraxon_maxdf.csv',
    'AxonVDR_region':       RESULTS_DIR / 'axon_maxdf.csv',
}[REGION]


#### Create a list to hold the subplots

In [ ]:
subplots = []

## Pick a neuron to simulate

In [ ]:
neuronType = 'DNa02'#'hDeltaA'#'PEN_a(PEN1)'
criteria = NC(type=neuronType) 
neuron_df, roi_counts_df = fetch_neurons(criteria)

bodyid = neuron_df['bodyId'][0]
neuronToSim = bodyid


# neuronToSim = 1140245595 # DNa02 ID

## Pull the skeleton and synapses

In [ ]:
# Get its skeleton
s = c.fetch_skeleton(neuronToSim, format='pandas')
s['bodyId'] = neuronToSim 
s['color'] = 'black'
segments = s.merge(s, 'inner',
                   left_on=['bodyId', 'rowId'],
                   right_on=['bodyId', 'link'],
                   suffixes=['_child', '_parent'])

# Get full list of input synapse properties
inputs = fetch_synapse_connections(None, neuronToSim, client=c)

# Get the name and type of the upstream partners and add them to the inputs object
neuronToSim_df, connToSim_df = fetch_adjacencies(None, neuronToSim)
connToSim_df = merge_neuron_properties(neuronToSim_df, connToSim_df, ['type', 'instance'])
inputs['type'] = [connToSim_df[connToSim_df['bodyId_pre'] == i].type_pre.values[0] for i in inputs['bodyId_pre']]

inputs['instance'] = [connToSim_df[connToSim_df['bodyId_pre'] == i].instance_pre.values[0] for i in inputs['bodyId_pre']]

# Find the nearest node for each synapse
inputs['coords'] = list(zip(inputs["x_post"], inputs["y_post"], inputs["z_post"]))
tree = cKDTree(list(zip(s["x"], s["y"], s["z"])))
inputs['swcid'] = inputs['coords'].apply(lambda x: tree.query(x)[1]+1)

# Get full list of output synapse properties
outputs = fetch_synapse_connections(neuronToSim, client=c)

# Get the name and type of the downstream partners and add them to the outputs object
neuronToMon_df, connToMon_df = fetch_adjacencies(neuronToSim,None)
connToMon_df = merge_neuron_properties(neuronToMon_df, connToMon_df, ['type', 'instance'])
outputs['type'] = [connToMon_df[connToMon_df['bodyId_post'] == i].type_post.values[0] for i in outputs['bodyId_post']]
outputs['instance'] = [connToMon_df[connToMon_df['bodyId_post'] == i].instance_post.values[0] for i in outputs['bodyId_post']]

# Find the nearest node for each synapse
outputs['coords'] = list(zip(outputs["x_pre"], outputs["y_pre"], outputs["z_pre"]))
tree = cKDTree(list(zip(s["x"], s["y"], s["z"])))
outputs['swcid'] = outputs['coords'].apply(lambda x: tree.query(x)[1]+1)

In [ ]:
print(inputs.head())

### Plot

In [ ]:
def plotSkeleton(segments, panel):
    print("plotting skeleton:")
    for index, seg in tqdm(segments.iterrows(), total=len(segments)):
        panel.plot([seg['x_child'], seg['x_parent']],
                   [seg['z_child'], seg['z_parent']],
                   marker='o', 
                   markersize=0,
                   linewidth=0.25,
                   markeredgewidth=0,
                   color = (0.7, 0.7, 0.7),
                   )

In [ ]:
figureWidth = 6
figureHeight = 6
panelWidth = 4
panelHeight = 4
plt.figure(figsize=(figureWidth, figureHeight))
panel = plt.axes([0.1, 0.1, panelWidth/figureWidth, panelHeight/figureHeight])

point_limit = 1000

for index, seg in tqdm(segments.iterrows(), total=len(segments)):
    panel.plot([seg['x_child'], seg['x_parent']],
               [seg['z_child'], seg['z_parent']],
               marker='o', 
               markersize=0,
               linewidth=0.25,
               markeredgewidth=0,
               color = 'black',
               )

for index, syn in tqdm(inputs.iterrows(), total=len(inputs)):
    panel.plot(syn['x_post'], syn['z_post'],
               marker='o', 
               markersize=0.5,
               linewidth=0,
               markeredgewidth=0,
               color = 'magenta',
               )


plt.gca().invert_yaxis()
plt.gca().set_aspect('equal')
plt.savefig("DNa02_Skeleton_x_z_Axis.png", dpi=600)
plt.show()

In [ ]:
dna02 = neu.fetch_skeletons(neu.SegmentCriteria(instance='DNa02_R', regex=True), with_synapses = True, heal = True)
dna02[0]

In [ ]:
#dna02[0]?

In [ ]:
# Load example neurons
nl =dna02

# Convert example neurons from voxels to nanometers
nl_nm = nl.convert_units("nm")

# Reroot to soma
#nl_nm[nl_nm.soma != None].reroot(nl_nm[nl_nm.soma != None].soma, inplace=True)

highlight = nl_nm.connectors.connector_id[100:150]

In [ ]:
#navis.plot_flat(nl_nm[0], layout="subway", shade_by_length = True,  highlight_connectors=highlight)

In [ ]:
dna02[0].branch_points

In [ ]:
# # Get all branch points from the node table
# bp = dna02.nodes
# bp.head()

In [ ]:
ln1 = navis.longest_neurite(dna02[0], n=1, reroot_soma=True)

In [ ]:
ln4 = navis.longest_neurite(dna02[0], n=4, reroot_soma=True)

In [ ]:
ln1.nodes

In [ ]:
# Create a boolean mask for the condition
condition = (ln1.nodes['y'] >= 17000) & (ln1.nodes['y'] <= 22000)

# Use loc with the boolean mask to select the rows
bb = ln1.nodes[condition]

In [ ]:
# Create a boolean mask for the condition
condition2 = (ln4.nodes['y'] <= 15000)
# Use loc with the boolean mask to select the rows
bb2 = ln4.nodes[condition2]

In [ ]:
navis.plot2d([ln1, bb])

In [ ]:
#navis.plot3d([ln4, bb])

In [ ]:
navis.plot2d([ln4, bb])

In [ ]:
condition2 = (
    (ln4.nodes['x'] >= 18000) & (ln4.nodes['x'] <= 22000) &
    (ln4.nodes['y'] < 15000) &
    (ln4.nodes['z'] > 30000) &
    ~((ln4.nodes['y'] > 10000) & (ln4.nodes['x'] > 19000)) &
    ~(
        (ln4.nodes['z'] >= 30000) & (ln4.nodes['z'] <= 32500) &
        (ln4.nodes['y'] >= 10000) & (ln4.nodes['y'] <= 12500)
    )
)

# Apply the condition
bb2 = ln4.nodes.loc[condition2]

In [ ]:
# Remove the first 3 nodes
bbfront = bb2.iloc[3:].copy()
axonnav = bbfront

In [ ]:
navis.plot3d([ln4, bbfront])

In [ ]:
#combine the IDR NaV site witht the Axonal NaV site to represent the full tract of active conductance
#Leave them seperate, and use just the bbfront to model the axonal site exclusively
# combined_bb = pd.concat([bbfront, bb], ignore_index=True)
# nav = combined_bb

In [ ]:
navis.plot3d([dna02, bbfront])

In [ ]:
#central_vdr_nodes = combined_bb.nodes

In [ ]:
#central_vdr_nodes.node_id.unique()

In [ ]:
#mon_list = [5625, 5645, 5766, 5771, 5790, 5792, 5793, 5795, 5797, 5799, 5801,
#       5803, 5805, 5807, 5808, 5810, 5812, 5813, 5814, 5818, 5820, 5822,
#       5824]

In [ ]:
# #nl_nm[0].nodes['hover_id'] = nl_nm[0].nodes['node_id'].astype(str)

# # Plot on 2d
# navis.plot2d([dna02, nav], color_by = 'strahler_index', palette="plasma", connectors = False)


In [ ]:
inputs['instance']

## Choose the synapses that will receive input and where the output will be monitored

#### Choose which synapses will receive an EPSP

In [ ]:
#in_instance = 'PFL3(PB12c)_R1_C2_irreg'
# in_instance = inputs['instance']

# all_input_syns = inputs[inputs['instance'] == in_instance]

# index = 1

# in_name = in_instance + '_point_' + str(index)

# input_syns = inputs

#input_syns = inputs[(inputs['roi_post'] == 'PB') & (inputs['instance_pre'] == 'EPG(PB08)_L7')]

# drive_list = input_syns.swcid.values

input_syns = inputs  # includes all input synapses to DNa02
drive_list = input_syns['swcid'].values  # list of all input swcids

In [ ]:
print(input_syns)
# for d in drive_list:
#     print(d)

In [ ]:
#nav

#### Choose which output synapses to monitor

In [ ]:
# out_instance = '5813024722'
#out_postID = ['5813024722'
out_name = REGION

mon_list = bbfront['node_id'].unique().tolist()
#output_syns = outputs[(outputs['bodyId_post'] == out_postID) ]
#output_syns = inputs.iloc[[4896]] #this is the lower vdr node [4896 for swcid 7650

#output_syns = outputs[(outputs['roi_post'] == 'EB') & (outputs['instance_post'] == 'EPG(PB08)_R3')]
#mon_list = output_syns.swcid.values
#mon_list = "7650"
output_syns = mon_list

In [ ]:
#outputs['swcid'].sort_values()

In [ ]:
#print(outputs)
print(output_syns)

In [ ]:
# from bokeh.plotting import figure, show
# from bokeh.models import ColumnDataSource, HoverTool, Label
# import numpy as np

# # # Ensure segments_source is defined before using it
# segments_source = ColumnDataSource(data=dict(
#     rowId_child=segments['rowId_child'],
#     x_child=segments['x_child'],
#     y_child=segments['y_child'],
#     z_child=segments['z_child'],
#     radius_child=segments['radius_child'],
#     link_child=segments['link_child'],
#     bodyId=segments['bodyId'],
#     color_child=segments['color_child'],
#     rowId_parent=segments['rowId_parent'],
#     x_parent=segments['x_parent'],
#     y_parent=segments['y_parent'],
#     z_parent=segments['z_parent'],
#     radius_parent=segments['radius_parent'],
#     link_parent=segments['link_parent']
# ))

# # Create the figure
# p = figure(match_aspect=True,
#            outer_width=300,
#            outer_height=400,
#            title="DNa02")

# # Create the segment glyph and store the renderer
# segment_renderer = p.segment(x0='x_child', x1='x_parent',
#                              y0='z_child', y1='z_parent',
#                              color='color_child',
#                              source=segments_source)  # Now segments_source is properly defined

# # Create a ColumnDataSource for the scatter plot
# source = ColumnDataSource(data=dict(
#     x_pre=inputs['x_pre'],
#     z_pre=inputs['z_pre'],
#     instance=inputs['instance'],
#     swcid=inputs['swcid']
# ))

# # Scatter plot with tooltips
# scatter = p.scatter('x_pre', 'z_pre',
#                     color='green', size=3, alpha=1,
#                     legend_label="pre", source=source)


# #highlight = p.scatter('x_pre', 'z_pre',
#                       #color='red', size=10, alpha=1,
#                       #legend_label="Regional API VDR, Central", source=output_syns)


# # Add text label
# out_text = Label(x=np.max(inputs['x_pre']),
#                  y=np.min(inputs['z_pre']),
#                  text="Output Instance",
#                  text_color='green')
# p.add_layout(out_text)

# # Add hover tool for scatter plot (pre data)
# hover_scatter = HoverTool(renderers=[scatter], 
#                           tooltips=[("Instance", "@instance"),
#                                     ("SWCID", "@swcid")])
# p.add_tools(hover_scatter)

# # Add hover tool for segments (child-parent connections)
# hover_segments = HoverTool(renderers=[segment_renderer],  # Explicitly reference the segment renderer
#                            tooltips=[("Child ID", "@rowId_child"),
#                                      ("Parent ID", "@rowId_parent"),
#                                      ("Body ID", "@bodyId"),
#                                      ("Radius Child", "@radius_child"),
#                                      ("Radius Parent", "@radius_parent")])
# p.add_tools(hover_segments)

# # Additional lines
# p.line([np.min(segments['x_child'])-1000, np.min(segments['x_child'])],
#        [np.max(segments['z_child'])+1000, np.max(segments['z_child'])+1000])
# p.line([np.min(segments['x_child'])-1000, np.min(segments['x_child'])-1000],
#        [np.max(segments['z_child']), np.max(segments['z_child'])+1000])

# # Formatting
# p.y_range.flipped = True
# p.xgrid.grid_line_color = None
# p.ygrid.grid_line_color = None
# p.axis.visible = False

# show(p)
# subplots.append(p)

In [ ]:
# #insert the swcid of the node you want to use as the ouput
# x = 7650
# print(input_syns[input_syns['swcid'] == x].index)

In [ ]:
# #paste the outout of the previous cell into this iloc to find yournode
# inputs.iloc[[4896]]

In [ ]:
output_syns_df = axonnav

### Plot those synapses on the skeleton

In [ ]:
p = figure(match_aspect=True,
           outer_width=300,
           outer_height=400,
           title = "DNa02")


p.segment(x0='x_child', x1='x_parent',
          y0='z_child', y1='z_parent',
          color='color_child',
          source=segments)
p.scatter(input_syns['x_post'], input_syns['z_post'],
         color='magenta',size=.9,alpha=1,
         legend_label="Input Synapse")
#in_text = bokeh.models.Label(x=np.max(input_syns['x_post']),
#                             y=np.min(input_syns['z_post']),
#                             text=in_instance,
#                             text_color = 'magenta')
# p.add_layout(in_text)
p.scatter(output_syns_df['x'], output_syns_df['z'],
         color='green',size=3,alpha=2,
        legend_label="Chosen Output")
# out_text = bokeh.models.Label(x=np.max(output_syns_df['x_pre']),
#                               y=np.min(output_syns_df['z_pre']),
#                               text=str(x),
#                               text_color = 'green')
#p.add_layout(out_text)

p.line([np.min(segments['x_child'])-1000,np.min(segments['x_child'])],
       [np.max(segments['z_child'])+1000,np.max(segments['z_child'])+1000])
p.line([np.min(segments['x_child'])-1000,np.min(segments['x_child'])-1000],
       [np.max(segments['z_child']),np.max(segments['z_child'])+1000])       

p.y_range.flipped = True
p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = None
p.axis.visible = False





show(p)

subplots.append(p)

## Setup a passive model for one neuron using the cable equations

#### R and C values from Scheffer et al.

![title](RandCValues.png)

In [ ]:
# Axon resistance.
Ra_LOW = 0.4
Ra_MED=1.2
Ra_HIGH=4.0

# Membrane resistance.
Rm_LOW = 0.2
Rm_MED=0.8
Rm_HIGH=3.11

nm = NeuronModel(neuron_df['bodyId'][0],Ra=Ra_MED, Rm=Rm_MED, Cm=1e-2)

In [ ]:
print(nm.skeleton_df[nm.skeleton_df['link'] == -1])

In [ ]:
if RUN_SIMULATION:
    if not (SPICE_FOLDER and os.path.isdir(SPICE_FOLDER)):
        raise RuntimeError("RUN_SIMULATION is True but NGSPICE_BIN is not set "
                           "to a valid ngspice bin directory.")

    # Region-tagged AND timestamped: the IDR and axon notebooks write to
    # distinct filenames, and a re-run never clobbers an earlier sweep.
    timestr = time.strftime("%Y%m%d-%H%M%S")
    data_file = RESULTS_DIR / (
        f"{neuronType}-{bodyid}_All_ins-_outs-{out_name}_{timestr}.tsv")
    print(f"writing sweep to {data_file}")
    with open(data_file, 'a') as file:
        file.write("combo_id\tin_swcid\tin_voltage\tin_time"
                   "\tout_swcid\tout_voltage\tout_time")

    for i in tqdm(range(len(input_syns))):
        drive_list = input_syns[i:i+1].swcid.values

        #### Add the voltage source and link it to the specified inputs 

        syn_conductance_Lou = 10000000000
        syn_conductance_Kris_nACh = 30000000000

        rise_tau_Lou = 0.1
        rise_tau_Kris_nACh = 0.6

        decay_tau_Lou = 1.1
        decay_tau_Kris_nACh = 2.1

        syn_conductance = syn_conductance_Kris_nACh
        rise_tau = rise_tau_Kris_nACh
        decay_tau = decay_tau_Kris_nACh

        drive_str = ''

        # Add a resistor between the node and the voltage source
        r = 1
        for drive in tqdm(drive_list):
            # apply current at the specified input location
            drive_str += f"RDRIVE{r} {drive} {len(nm.skeleton_df)+1} {syn_conductance}\n" # 0.1 ns conductance
            r = r + 1

        # Add the voltage source
        drive_str += f"V1 {len(nm.skeleton_df)+1} 0 EXP(0 60.0 0.1 {rise_tau} 1.1 {decay_tau} 40)\n"

        # Run a transient analysis
        drive_str += ".tran 0.1 40\n" # work from 0-10 ms (try 40)

        ## Run the simulation and parse the output
        ### *Note for Windows users:* Change the directory in os.chdir to point to the location of your ngspice executable. Also, change 'set filetype=ascii' in Spice64/share/ngspice/scripts/spinit to 'set filetype=binary'
        ### *Note for Mac and Linux users:*  You may remove the os.chdir line. Replace "ngspice_con" with "ngspice"

        # Call command line spice simulator and write to temporary file
        fd, path = mkstemp()

        # Switch the current directory to the one where ngspice is stored
        CWD = os.getcwd()
        os.chdir(SPICE_FOLDER)

        # Run the spice model
        p = Popen(["ngspice_con", "-b", "-r", path], stdin=PIPE, stdout=DEVNULL, stderr=DEVNULL)
        data = nm.spice_model + drive_str
        p.stdin.write(data.encode())
        p.stdin.close()
        p.wait()

        # Return to the original directory
        os.chdir(CWD)

        """Read ngspice binary raw files. Return tuple of the data, and the
        plot metadata. The dtype of the data contains field names. This is
        not very robust yet, and only supports ngspice.
        # Example header of raw file
        # Title: rc band pass example circuit
        # Date: Sun Feb 21 11:29:14  2016
        # Plotname: AC Analysis
        # Flags: complex
        # No. Variables: 3
        # No. Points: 41
        # Variables:
        #         0       frequency       frequency       grid=3
        #         1       v(out)  voltage
        #         2       v(in)   voltage
        # Binary:
        """
        sim_results = None
        BSIZE_SP = 512 # Max size of a line of data; we don't want to read the
                        # whole file to find a line, in case file does not have
                        # expected structure.
        MDATA_LIST = [b'title', b'date', b'plotname', b'flags', b'no. variables', b'no. points', b'dimensions', b'command', b'option']
        with os.fdopen(fd, 'rb') as fp:
            plot = {}
            count = 0
            arrs = []
            plots = []
            while (True):
                try:
                    mdata = fp.readline(BSIZE_SP).split(b':', maxsplit=1)
                except:
                    raise RuntimeError("cannot parse spice output")
                if len(mdata) == 2:
                    if mdata[0].lower() in MDATA_LIST:
                        plot[mdata[0].lower()] = mdata[1].strip()
                    if mdata[0].lower() == b'variables':
                        nvars = int(plot[b'no. variables'])
                        npoints = int(plot[b'no. points'])
                        plot['varnames'] = []
                        plot['varunits'] = []
                        for varn in range(nvars):
                            varspec = (fp.readline(BSIZE_SP).strip()
                                       .decode('ascii').split())
                            assert(varn == int(varspec[0]))
                            plot['varnames'].append(varspec[1])
                            plot['varunits'].append(varspec[2])
                    if mdata[0].lower() == b'binary':
                        rowdtype = np.dtype({'names': plot['varnames'],
                                             'formats': [np.complex128 if b'complex'
                                                         in plot[b'flags']
                                                         else np.float64]*nvars})
                        # We should have all the metadata by now
                        arrs.append(np.fromfile(fp, dtype=rowdtype, count=npoints))
                        plots.append(plot)
                        fp.readline() # Read to the end of line
                else:
                    break

            # only one analysis
            sim_results = arrs[0]
        # Delete file
        os.unlink(path)

        # find input peak
        in_swcid  = drive_list[0]
        in_index  = np.argmax(sim_results[f'v({in_swcid})'])
        in_voltage = sim_results[f'v({in_swcid})'][in_index]
        in_time   = sim_results['time'][in_index]

        # loop over every output site
        for out_swcid in mon_list:
            out_index   = np.argmax(sim_results[f'v({out_swcid})'])
            out_voltage = sim_results[f'v({out_swcid})'][out_index]
            out_time    = sim_results['time'][out_index]

            # append one line per monitor with combo_id
            combo_id = f"{in_swcid}_to_{out_swcid}"
            with open(data_file, 'a') as file:
                file.write(
                    f"\n{combo_id}\t{in_swcid}\t{in_voltage}\t{in_time}"
                    f"\t{out_swcid}\t{out_voltage}\t{out_time}"
                )
else:
    # Newest completed sweep for THIS region, else the committed reduced CSV.
    prior = sorted(RESULTS_DIR.glob(
        f"*_All_ins-_outs-{out_name}_*.tsv"))
    data_file = prior[-1] if prior else COMMITTED_RESULTS
    print(f"RUN_SIMULATION is False -- reading {data_file}")


In [ ]:
# print(drive_str)

In [ ]:
# import jsonpickle
# import json

# serialized = jsonpickle.encode(sim_results[0])
# with open('sim_results_dump', 'w+') as file:
#     file.write(json.dumps(json.loads(serialized), indent=2))
    
# with open('skeleton_dump', 'w+') as file:
#     serialized_skeleton = jsonpickle.encode(s)
#     file.write(json.dumps(json.loads(serialized_skeleton), indent=2))

In [ ]:
# print(os.getcwd())

In [ ]:
# data_file = r'DNa02-1140245595_All_ins-_outs-central_VDR_region_20250718-171609.txt'

In [ ]:
# print(type(output_syns))
# print(output_syns)

In [ ]:
figureWidth = 4.5
figureHeight = 6
panelWidth = 3
panelHeight = 5
sideWidth = 0.25
sideHeight = 5
spacing = 0.05

plt.figure(figsize=(figureWidth, figureHeight))

# Plot the voltage decay at each input
# Read the sweep output by column NAME. The file's first column is combo_id,
# so the original positional parse mapped every field one slot to the left.
_cols = ['in_swcid', 'in_voltage', 'in_time', 'out_swcid', 'out_voltage', 'out_time']
data_list = pd.read_csv(data_file, sep='\t')[_cols].to_dict('records')

volt_ratio_list = [data['in_voltage'] / data['out_voltage'] for data in data_list]

palette = [(int(color), 255-int(color), 255) for color in np.linspace(0, 255, 256)]
def volt_to_color(volt_ratio, volt_ratio_list):
    norm_ratio = (volt_ratio - min(volt_ratio_list))/(max(volt_ratio_list) - min(volt_ratio_list))
    return (norm_ratio, 1-norm_ratio, 1)
        

panel_main = plt.axes([spacing, spacing, panelWidth/figureWidth, panelHeight/figureHeight])

# Plot skeleton
plotSkeleton(segments, panel_main)

# Plot input synapses
print("plotting synapses:")
for input_data in tqdm(data_list):
    syns = input_syns[input_syns['swcid'] == input_data['in_swcid']]
    volt_ratio = input_data['in_voltage'] / input_data['out_voltage']
    panel_main.plot(syns['x_post'], syns['z_post'],
                marker='o', 
                markersize=0.6,
                linewidth=0,
                markeredgewidth=0,
                color = volt_to_color(volt_ratio, volt_ratio_list),
                )

# Plot output synapse
panel_main.plot(output_syns_df['x'], output_syns_df['z'],
                marker='o', 
                markersize=0.6,
                linewidth=0,
                markeredgewidth=0,
                color='green')

panel_main.text(output_syns_df['x'].values[0], output_syns_df['z'].values[0],
                '   Output',
                fontsize=5,
                ha='left',
                va='center',
                color='green')

# Plot Scale Bar
scalexpos = 12000
scaleypos = 38000
scalewidth = 20000/8 # width in micrometers
scaleheight = 20

scalebar = mplpatches.Rectangle([scalexpos, scaleypos],
                                 scalewidth, scaleheight,
                                 linewidth=0,
                                 facecolor='black')
panel_main.add_patch(scalebar)

panel_main.text(scalexpos+scalewidth/2, scaleypos-10,
                "{:g}".format(scalewidth*8/1000)+"μm",
                fontsize=5,
                ha = 'center',
                va = 'bottom',
                color = 'black',
                )

panel_main.invert_yaxis()
panel_main.set_aspect('equal')
panel_main.tick_params(bottom=False, labelbottom=False,
                    left=False, labelleft=False,
                    right=False, labelright=False,
                    top=False, labeltop=False,)

# Plot Colormap
panel_side = plt.axes([spacing + panelWidth/figureWidth + spacing, spacing,
                       sideWidth/figureWidth, sideHeight/figureHeight])

for i in np.linspace(min(volt_ratio_list), max(volt_ratio_list), 256):
    space = (max(volt_ratio_list) - min(volt_ratio_list))/255
    rectangle = mplpatches.Rectangle([0, i], 1, space,
                                     linewidth=0,
                                     facecolor=volt_to_color(i, volt_ratio_list))
    panel_side.add_patch(rectangle)


panel_side.set_ylabel('Voltage Decay Ratio',
                       labelpad = -25,
                       rotation = 270)
panel_side.set_xlim([0, 1])
panel_side.set_ylim([min(volt_ratio_list), max(volt_ratio_list)])
panel_side.set_yticks([min(volt_ratio_list), max(volt_ratio_list)])
panel_side.tick_params(bottom=False, labelbottom=False,
                      left=False, labelleft=False,
                      right=True, labelright=True,
                      top=False, labeltop=False,)

## Look at the simulation output

#### Choose times to look at the voltage across the skeleton

In [ ]:
#tsToPlot = [0.01, 0.1, 0.25, 0.55, 1.1, 2, 4, 6, 8, 10, 12, 14, 16, 32]


tsToPlot = np.linspace(0, 15, 60).tolist()

In [ ]:
#tsToPlot

#### Plot the voltage profile of the outputs as compared to the inputs

In [ ]:
# input vs output voltage traces
if RUN_SIMULATION:
    tPts = sim_results["time"]

    p_outs = figure(outer_width=300, outer_height=400)

    # Plot the voltage coming in
    for d in drive_list:
        in_voltage = sim_results[f"v({d})"]
        p_outs.line(tPts,in_voltage,color='magenta')

    # Plot the voltage at the chosen synapses
    for m in mon_list:
        out_voltage = sim_results[f"v({m})"]
        p_outs.line(tPts,out_voltage,color='green')

    for t in tsToPlot:
        vline = bokeh.models.Span(location=t, dimension='height', line_color='gray', line_dash="dashed")
        p_outs.add_layout(vline)

    p_outs.xaxis.axis_label = 'time (ms)'
    p_outs.yaxis.axis_label = 'voltage (mV)'
    subplots.append(p_outs)
else:
    print("skipped: needs an in-memory ngspice run "
          "(set RUN_SIMULATION = True)")


In [ ]:
# show the trace figure
if RUN_SIMULATION:
    show(p_outs)
else:
    print("skipped: needs an in-memory ngspice run "
          "(set RUN_SIMULATION = True)")


#### Look at the voltage across the arbors at the chosen time points

In [ ]:
# voltage across the arbor per timepoint
if RUN_SIMULATION:
    # Specify a color palette for the voltage
    cols = bokeh.palettes.Turbo256

    VVals = np.linspace(0,max(in_voltage),num=len(cols))

    for t in tsToPlot:
        # Get the location of the timepoint in the the simulation output array
        tPt = np.abs(tPts - t).argmin()

        #Get the voltage at each point in the arbor at that time
        sTmp = s
        sTmp['voltage'] = [0.0] * len(sTmp)
        for i in range(len(sTmp)):
            sTmp.loc[i,'voltage'] = sim_results[tPt][i+1]

        #Convert voltage to color
        colVals = [''] * len(sTmp)
        for v in range(0,len(sTmp)):
            colVals[v] = cols[np.abs(VVals - sTmp['voltage'][v]).argmin()]
        sTmp['voltage color'] = colVals

        # Create segments from the skeleton
        segments = sTmp.merge(s, 'inner',
                              left_on=['bodyId', 'rowId'],
                              right_on=['bodyId', 'link'],
                              suffixes=['_child', '_parent'])

        #Plot the skeleton
        p = figure(match_aspect=True,
                  outer_width=300, outer_height=400)

        p.segment(x0='x_child', x1='x_parent',
                  y0='z_child', y1='z_parent',
                  color='voltage color_child',
                  source=segments)
        #p.scatter(input_syns['x_post'], input_syns['z_post'],
        #     color='magenta',size=.1,alpha=0.5)

        p.y_range.flipped = True
        p.xgrid.grid_line_color = None
        p.ygrid.grid_line_color = None
        p.axis.visible = False
        p.title.text = f'time = {t}ms'

        subplots.append(p)
else:
    print("skipped: needs an in-memory ngspice run "
          "(set RUN_SIMULATION = True)")


In [ ]:
xrange = 20
subS = s[(s['x'] >= 19377-xrange) & (s['x'] <= 19377+xrange) & (s['y'] >= 18529-xrange) & (s['y'] <= 18529+xrange)]
print(subS)
# print(len(s))

## Display and save the plots

In [ ]:
# output_file(neuronType + '-' + str(bodyid) + '_' + 'All_ins-' + 'central_region_outs-' + out_name + ".html")
grid = gridplot(subplots,ncols=4)
show(grid)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from bokeh.palettes import inferno

In [ ]:
print(os.getcwd())

In [ ]:
# animation over the timepoints
if RUN_SIMULATION:
    # Create a colorbar axis on the right side of the plot
    cax = fig.add_axes([0.85, 0.1, 0.05, 0.8])

    def init():
        ax.clear()
        return ax,

    def update(t):
        # Get the location of the timepoint in the simulation output array
        tPt = np.abs(tPts - t).argmin()
    
        # Get the voltage at each point in the arbor at that time
        sTmp = s.copy()
        sTmp['voltage'] = [0.0] * len(sTmp)
        for i in range(len(sTmp)):
            sTmp.loc[i, 'voltage'] = sim_results[tPt][i+1]
    
        # Convert voltage to color
        colVals = [''] * len(sTmp)
        for v in range(len(sTmp)):
            colVals[v] = cols[np.abs(VVals - sTmp['voltage'][v]).argmin()]
        sTmp['voltage color'] = colVals

        # Create segments from the skeleton
        segments = sTmp.merge(s, 'inner',
                              left_on=['bodyId', 'rowId'],
                              right_on=['bodyId', 'link'],
                              suffixes=['_child', '_parent'])

        # Plot the skeleton
        ax.clear()
        scatter = ax.scatter(segments['x_child'], segments['z_child'], color=segments['voltage color_child'])
        #ax.scatter(input_syns['x_post'], input_syns['z_post'], color='magenta', s=1, alpha=0.2,)
        ax.set_title(f'time = {t}ms')
    
        # Adjust plot scaling, axis, and viewpoint
        ax.set_aspect('equal')
        ax.set_xlim([min(segments['x_child']), max(segments['x_child'])])
        ax.set_ylim([min(segments['z_child']), max(segments['z_child'])])
    
        # Invert the y-axis to flip the image vertically
        ax.invert_yaxis()
    
        ax.grid(False)
        ax.axis('off')
    
        return scatter,

    # Create a colorbar based on the voltage values and color palette
    norm = Normalize(vmin=min(VVals), vmax=max(VVals))
    sm = ScalarMappable(norm=norm, cmap='turbo')

    sm.set_array([])
    fig.colorbar(sm, cax=cax)
    timestr = time.strftime("%Y%m%d-%H%M%S")


    ani = FuncAnimation(fig, update, frames=tsToPlot, init_func=init, blit=False)
    # ani.save(timestr + '_animation.gif')
    # gif_file = (timestr + '_animation.gif')
else:
    print("skipped: needs an in-memory ngspice run "
          "(set RUN_SIMULATION = True)")


In [ ]:
# # # Display the animation in the notebook
# # from IPython.display import Image
# # Image(gif_file)

In [ ]:
# gif_file

In [ ]:
# ani.save(timestr + '_animation.gif')
